# Briefcase AI Telemetry SDK - End-to-End Demo

This notebook demonstrates a complete production workflow using the Briefcase AI Telemetry SDK with:

- **LangChain Integration**: Monitor LangChain agents and chains
- **Replay System**: Capture and replay AI interactions for testing
- **Prometheus Metrics**: Export telemetry data for monitoring
- **Grafana Dashboard**: Visualize AI performance metrics
- **P95 Benchmarking**: Performance analysis and optimization

## Prerequisites

Install required packages:

In [ ]:
# Install required packages
!pip install briefcase-ai-telemetry-sdk langchain langchain-openai prometheus_client pandas matplotlib numpy seaborn

## 📦 Setup and Imports

In [ ]:
import os
import json
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Any
from dataclasses import dataclass

# Briefcase AI Telemetry
import briefcase_ai_telemetry as bt

# LangChain imports
from langchain.agents import initialize_agent, AgentType
from langchain.tools import BaseTool
from langchain.schema import BaseMessage
from langchain.callbacks.base import BaseCallbackHandler
from langchain_openai import ChatOpenAI

# Prometheus monitoring
from prometheus_client import Counter, Histogram, Gauge, generate_latest, CONTENT_TYPE_LATEST

print("✅ All imports successful!")

## 🔧 Configuration and Setup

In [ ]:
# Configuration
DEMO_MODE = True  # Set to False for production with real API keys
OPENAI_API_KEY = "demo-key" if DEMO_MODE else os.getenv("OPENAI_API_KEY")
BRIEFCASE_API_KEY = "demo-briefcase-key"

# Initialize Briefcase AI Telemetry
client = bt.create_client(BRIEFCASE_API_KEY, enabled=not DEMO_MODE)
client.start_background_flush()

print(f"🚀 Briefcase AI Telemetry initialized (demo_mode={DEMO_MODE})")
print(f"📊 Client buffer size: {client.buffer_size()}")

## 📈 Prometheus Metrics Setup

Define Prometheus metrics for monitoring AI system performance:

In [ ]:
# Prometheus metrics
ai_requests_total = Counter(
    'ai_requests_total',
    'Total number of AI requests',
    ['model', 'status', 'agent_type']
)

ai_request_duration = Histogram(
    'ai_request_duration_seconds',
    'AI request duration in seconds',
    ['model', 'agent_type'],
    buckets=[0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 30.0, 60.0]
)

ai_cost_total = Counter(
    'ai_cost_usd_total',
    'Total AI cost in USD',
    ['model', 'agent_type']
)

ai_tokens_total = Counter(
    'ai_tokens_total',
    'Total tokens processed',
    ['model', 'token_type']  # input/output
)

ai_drift_score = Gauge(
    'ai_drift_consistency_score',
    'AI model consistency score (0-100)',
    ['model', 'agent_type']
)

ai_accuracy_score = Gauge(
    'ai_accuracy_score',
    'AI accuracy score (0-1)',
    ['model', 'agent_type']
)

print("📊 Prometheus metrics initialized")

## 🎬 Replay System Implementation

Create a system to capture and replay AI interactions for testing and validation:

In [ ]:
@dataclass
class AIInteraction:
    """Represents a single AI interaction for replay."""
    timestamp: datetime
    agent_id: str
    input_text: str
    expected_output: str
    actual_output: Optional[str] = None
    model: str = "gpt-3.5-turbo"
    metadata: Dict[str, Any] = None
    accuracy: Optional[float] = None
    cost: Optional[float] = None
    duration: Optional[float] = None

class ReplayLoader:
    """Loads and manages AI interaction replays."""
    
    def __init__(self, telemetry_client):
        self.telemetry_client = telemetry_client
        self.interactions: List[AIInteraction] = []
        self.replay_results: List[Dict] = []
    
    def add_interaction(self, interaction: AIInteraction):
        """Add an interaction to the replay dataset."""
        self.interactions.append(interaction)
    
    def load_from_json(self, file_path: str):
        """Load interactions from JSON file."""
        try:
            with open(file_path, 'r') as f:
                data = json.load(f)
            
            for item in data:
                interaction = AIInteraction(
                    timestamp=datetime.fromisoformat(item['timestamp']),
                    agent_id=item['agent_id'],
                    input_text=item['input_text'],
                    expected_output=item['expected_output'],
                    model=item.get('model', 'gpt-3.5-turbo'),
                    metadata=item.get('metadata', {})
                )
                self.add_interaction(interaction)
            
            print(f"✅ Loaded {len(self.interactions)} interactions from {file_path}")
        
        except FileNotFoundError:
            print(f"⚠️ File {file_path} not found. Creating sample interactions.")
            self._create_sample_interactions()
    
    def _create_sample_interactions(self):
        """Create sample interactions for demo purposes."""
        sample_interactions = [
            AIInteraction(
                timestamp=datetime.now() - timedelta(hours=1),
                agent_id="customer_support",
                input_text="How can I reset my password?",
                expected_output="To reset your password, please visit the login page and click 'Forgot Password'. You'll receive an email with reset instructions.",
                model="gpt-3.5-turbo",
                metadata={"priority": "high", "category": "account"}
            ),
            AIInteraction(
                timestamp=datetime.now() - timedelta(hours=2),
                agent_id="data_analyst",
                input_text="Analyze the sales trend for Q3 2024",
                expected_output="Q3 2024 sales show a 15% increase compared to Q2, with strongest growth in the technology and healthcare sectors.",
                model="gpt-4",
                metadata={"priority": "medium", "category": "analytics"}
            ),
            AIInteraction(
                timestamp=datetime.now() - timedelta(hours=3),
                agent_id="code_reviewer",
                input_text="Review this Python function for bugs",
                expected_output="The function looks good overall. Consider adding input validation and error handling for edge cases.",
                model="gpt-4",
                metadata={"priority": "low", "category": "code_review"}
            )
        ]
        
        self.interactions.extend(sample_interactions)
        print(f"✅ Created {len(sample_interactions)} sample interactions")
    
    def replay_interaction(self, interaction: AIInteraction) -> Dict:
        """Replay a single interaction and capture results."""
        start_time = time.time()
        
        # Create agent instrument for this interaction
        instrument = bt.create_agent_instrument(
            agent_id=hash(interaction.agent_id) % 10000,
            client=self.telemetry_client
        )
        
        session = instrument.start()
        
        try:
            # Simulate AI processing (in real scenario, this would be actual AI call)
            if DEMO_MODE:
                # Simulate processing time
                time.sleep(np.random.uniform(0.1, 0.5))
                
                # Add some variation to expected output for demo
                variations = [
                    interaction.expected_output,
                    interaction.expected_output.replace(".", "!"),
                    interaction.expected_output + " Let me know if you need more help."
                ]
                actual_output = np.random.choice(variations)
            else:
                # Real AI processing would go here
                actual_output = interaction.expected_output
            
            duration = time.time() - start_time
            
            # Calculate accuracy (similarity between expected and actual)
            accuracy = self._calculate_similarity(interaction.expected_output, actual_output)
            
            # Estimate cost
            cost_estimate = bt.estimate_cost(
                interaction.model,
                interaction.input_text,
                actual_output
            )
            cost = cost_estimate.total_cost if cost_estimate else 0.0
            
            # Update session
            session.set_input_output(interaction.input_text, actual_output)
            session.set_model_info(interaction.model, temperature=0.1)
            session.set_accuracy(accuracy)
            session.set_cost(cost)
            session.add_reasoning_step(f"Processed {interaction.agent_id} request")
            
            # Add metadata
            for key, value in (interaction.metadata or {}).items():
                session.set_metadata(key, str(value))
            
            session.set_metadata("replay_mode", "true")
            session.set_metadata("original_timestamp", interaction.timestamp.isoformat())
            
            # Update Prometheus metrics
            ai_requests_total.labels(
                model=interaction.model,
                status="success",
                agent_type=interaction.agent_id
            ).inc()
            
            ai_request_duration.labels(
                model=interaction.model,
                agent_type=interaction.agent_id
            ).observe(duration)
            
            ai_cost_total.labels(
                model=interaction.model,
                agent_type=interaction.agent_id
            ).inc(cost)
            
            if cost_estimate:
                ai_tokens_total.labels(
                    model=interaction.model,
                    token_type="input"
                ).inc(cost_estimate.input_tokens)
                
                ai_tokens_total.labels(
                    model=interaction.model,
                    token_type="output"
                ).inc(cost_estimate.output_tokens)
            
            ai_accuracy_score.labels(
                model=interaction.model,
                agent_type=interaction.agent_id
            ).set(accuracy)
            
            result = {
                "interaction": interaction,
                "actual_output": actual_output,
                "accuracy": accuracy,
                "cost": cost,
                "duration": duration,
                "status": "success"
            }
            
        except Exception as e:
            session.set_error(str(e))
            session.set_accuracy(0.0)
            
            ai_requests_total.labels(
                model=interaction.model,
                status="error",
                agent_type=interaction.agent_id
            ).inc()
            
            result = {
                "interaction": interaction,
                "error": str(e),
                "status": "error"
            }
        
        finally:
            session.finish()
        
        return result
    
    def _calculate_similarity(self, expected: str, actual: str) -> float:
        """Calculate similarity between expected and actual output."""
        # Simple similarity calculation (in production, use more sophisticated methods)
        expected_words = set(expected.lower().split())
        actual_words = set(actual.lower().split())
        
        if not expected_words:
            return 1.0 if not actual_words else 0.0
        
        intersection = expected_words.intersection(actual_words)
        union = expected_words.union(actual_words)
        
        return len(intersection) / len(union) if union else 1.0
    
    def run_replay_batch(self, max_interactions: Optional[int] = None) -> List[Dict]:
        """Run replay on a batch of interactions."""
        interactions_to_run = self.interactions[:max_interactions] if max_interactions else self.interactions
        
        print(f"🔄 Starting replay of {len(interactions_to_run)} interactions...")
        
        results = []
        for i, interaction in enumerate(interactions_to_run):
            print(f"  Processing {i+1}/{len(interactions_to_run)}: {interaction.agent_id}")
            result = self.replay_interaction(interaction)
            results.append(result)
        
        self.replay_results.extend(results)
        print(f"✅ Replay completed! {len(results)} interactions processed.")
        
        return results

# Initialize replay loader
replay_loader = ReplayLoader(client)
replay_loader.load_from_json("interactions.json")  # Will create sample data if file not found

print(f"📚 Replay system initialized with {len(replay_loader.interactions)} interactions")

## 🦜 LangChain Integration

Integrate Briefcase AI Telemetry with LangChain agents and tools:

In [ ]:
class TelemetryCallbackHandler(BaseCallbackHandler):
    """LangChain callback handler for Briefcase AI Telemetry."""
    
    def __init__(self, telemetry_client, agent_id: int):
        self.telemetry_client = telemetry_client
        self.agent_id = agent_id
        self.current_session = None
        self.start_time = None
        self.tool_calls = []
        self.reasoning_steps = []
    
    def on_chain_start(self, serialized, inputs, **kwargs):
        """Called when a chain starts."""
        self.start_time = time.time()
        
        # Create new instrumentation session
        instrument = bt.create_agent_instrument(self.agent_id, self.telemetry_client)
        self.current_session = instrument.start()
        
        # Extract input text
        input_text = inputs.get('input', '') or str(inputs)
        self.current_session.set_metadata("chain_type", serialized.get('name', 'unknown'))
        self.current_session.add_reasoning_step(f"Chain started: {serialized.get('name', 'unknown')}")
    
    def on_tool_start(self, serialized, input_str, **kwargs):
        """Called when a tool starts."""
        tool_name = serialized.get('name', 'unknown_tool')
        self.tool_calls.append({
            'tool': tool_name,
            'input': input_str,
            'start_time': time.time()
        })
        
        if self.current_session:
            self.current_session.add_reasoning_step(f"Using tool: {tool_name}")
            self.current_session.add_tool_call(tool_name, {'input': input_str[:100]})  # Truncate for storage
    
    def on_tool_end(self, output, **kwargs):
        """Called when a tool ends."""
        if self.tool_calls:
            tool_call = self.tool_calls[-1]
            tool_call['output'] = str(output)[:200]  # Truncate output
            tool_call['duration'] = time.time() - tool_call['start_time']
            
            if self.current_session:
                self.current_session.add_reasoning_step(
                    f"Tool {tool_call['tool']} completed in {tool_call['duration']:.2f}s"
                )
    
    def on_text(self, text, **kwargs):
        """Called when text is generated."""
        # Add reasoning step for significant text
        if self.current_session and len(text) > 10:
            self.reasoning_steps.append(text[:100])  # Store truncated reasoning
    
    def on_chain_end(self, outputs, **kwargs):
        """Called when a chain ends."""
        if not self.current_session:
            return
        
        duration = time.time() - self.start_time if self.start_time else 0
        
        # Extract output text
        output_text = outputs.get('output', '') or str(outputs)
        
        # Set session details
        self.current_session.set_model_info("langchain_agent", temperature=0.1)
        self.current_session.set_metadata("duration_seconds", str(duration))
        self.current_session.set_metadata("tools_used", str(len(self.tool_calls)))
        self.current_session.set_metadata("reasoning_steps", str(len(self.reasoning_steps)))
        
        # Add final reasoning steps
        for step in self.reasoning_steps[-3:]:  # Last 3 reasoning steps
            self.current_session.add_reasoning_step(step)
        
        # Calculate accuracy (simplified - in production, compare with expected output)
        accuracy = 0.8 + np.random.uniform(0, 0.2)  # Simulate 80-100% accuracy
        self.current_session.set_accuracy(accuracy)
        
        # Estimate cost based on complexity
        estimated_tokens = len(output_text.split()) * 1.3
        estimated_cost = estimated_tokens * 0.00002  # Rough estimate
        self.current_session.set_cost(estimated_cost)
        
        self.current_session.finish()
        
        # Update Prometheus metrics
        ai_requests_total.labels(
            model="langchain_agent",
            status="success",
            agent_type=f"agent_{self.agent_id}"
        ).inc()
        
        ai_request_duration.labels(
            model="langchain_agent",
            agent_type=f"agent_{self.agent_id}"
        ).observe(duration)
        
        ai_accuracy_score.labels(
            model="langchain_agent",
            agent_type=f"agent_{self.agent_id}"
        ).set(accuracy)
        
        # Reset for next chain
        self.current_session = None
        self.tool_calls = []
        self.reasoning_steps = []
    
    def on_chain_error(self, error, **kwargs):
        """Called when a chain errors."""
        if self.current_session:
            self.current_session.set_error(str(error))
            self.current_session.set_accuracy(0.0)
            self.current_session.finish()
        
        ai_requests_total.labels(
            model="langchain_agent",
            status="error",
            agent_type=f"agent_{self.agent_id}"
        ).inc()

# Custom LangChain tool for demonstration
class AnalyticsTool(BaseTool):
    name = "analytics_tool"
    description = "Useful for analyzing data and generating insights"
    
    def _run(self, query: str) -> str:
        # Simulate analytics processing
        time.sleep(0.2)
        
        analytics_responses = [
            f"Analysis shows a 15% increase in {query.split()[-1] if query.split() else 'metrics'}",
            f"Data indicates strong performance in {query.split()[-1] if query.split() else 'key areas'}",
            f"Trend analysis reveals growth patterns for {query.split()[-1] if query.split() else 'the dataset'}"
        ]
        
        return np.random.choice(analytics_responses)

class DatabaseTool(BaseTool):
    name = "database_tool"
    description = "Useful for querying databases and retrieving information"
    
    def _run(self, query: str) -> str:
        # Simulate database query
        time.sleep(0.1)
        
        db_responses = [
            "Query executed successfully. Found 42 matching records.",
            "Database returned 18 results for your query.",
            "Search completed. 7 relevant entries found."
        ]
        
        return np.random.choice(db_responses)

# Create LangChain agent with telemetry
def create_monitored_langchain_agent(agent_id: int):
    """Create a LangChain agent with telemetry monitoring."""
    
    # Initialize callback handler
    callback_handler = TelemetryCallbackHandler(client, agent_id)
    
    # Create tools
    tools = [AnalyticsTool(), DatabaseTool()]
    
    if DEMO_MODE:
        # Use a mock LLM for demo purposes
        class MockLLM:
            def __init__(self):
                self.callbacks = None
            
            def predict(self, text, callbacks=None):
                # Simulate LLM processing
                time.sleep(0.3)
                
                responses = [
                    "I'll help you with that analysis. Let me query the database first.",
                    "Based on the data, I can provide you with detailed insights.",
                    "Let me use the analytics tool to process this information."
                ]
                
                return np.random.choice(responses)
        
        llm = MockLLM()
        
        # Simple agent simulation
        class MockAgent:
            def __init__(self, llm, tools, callbacks):
                self.llm = llm
                self.tools = tools
                self.callbacks = callbacks
            
            def run(self, input_text):
                # Simulate agent execution
                if self.callbacks:
                    for callback in self.callbacks:
                        callback.on_chain_start({"name": "mock_agent"}, {"input": input_text})
                        
                        # Simulate tool usage
                        if "analyze" in input_text.lower():
                            callback.on_tool_start({"name": "analytics_tool"}, input_text)
                            tool_result = self.tools[0]._run(input_text)
                            callback.on_tool_end(tool_result)
                        
                        if "data" in input_text.lower():
                            callback.on_tool_start({"name": "database_tool"}, input_text)
                            tool_result = self.tools[1]._run(input_text)
                            callback.on_tool_end(tool_result)
                        
                        # Generate response
                        response = f"Based on the analysis, here's what I found: {self.llm.predict(input_text)}"
                        callback.on_text(response)
                        
                        callback.on_chain_end({"output": response})
                
                return f"Based on the analysis, here's what I found: {self.llm.predict(input_text)}"
        
        agent = MockAgent(llm, tools, [callback_handler])
    
    else:
        # Real LangChain agent (requires OpenAI API key)
        llm = ChatOpenAI(temperature=0.1, openai_api_key=OPENAI_API_KEY)
        agent = initialize_agent(
            tools,
            llm,
            agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
            callbacks=[callback_handler],
            verbose=True
        )
    
    return agent, callback_handler

print("🦜 LangChain integration completed")

## 🎯 P95 Benchmark Script

Performance benchmarking system to measure P95 latency and other key metrics:

In [ ]:
class P95Benchmark:
    """Performance benchmarking with P95 analysis."""
    
    def __init__(self, telemetry_client):
        self.telemetry_client = telemetry_client
        self.benchmark_results = []
        self.drift_history = []
    
    def run_performance_benchmark(self, 
                                 num_requests: int = 50,
                                 models: List[str] = None) -> Dict:
        """Run performance benchmark across multiple models."""
        
        if models is None:
            models = ["gpt-3.5-turbo", "gpt-4", "claude-3-sonnet"]
        
        print(f"🎯 Starting P95 benchmark with {num_requests} requests per model...")
        
        benchmark_data = {
            "timestamp": datetime.now(),
            "num_requests": num_requests,
            "models": {},
            "summary": {}
        }
        
        test_inputs = [
            "Explain quantum computing in simple terms",
            "Write a Python function to calculate fibonacci numbers",
            "Analyze the impact of AI on healthcare",
            "Summarize the key principles of machine learning",
            "Describe best practices for API design"
        ]
        
        for model in models:
            print(f"  Testing {model}...")
            model_results = self._benchmark_model(model, num_requests, test_inputs)
            benchmark_data["models"][model] = model_results
        
        # Calculate summary statistics
        benchmark_data["summary"] = self._calculate_summary_stats(benchmark_data["models"])
        
        self.benchmark_results.append(benchmark_data)
        
        print("✅ P95 benchmark completed!")
        return benchmark_data
    
    def _benchmark_model(self, model: str, num_requests: int, test_inputs: List[str]) -> Dict:
        """Benchmark a single model."""
        latencies = []
        costs = []
        accuracies = []
        outputs = []
        
        for i in range(num_requests):
            input_text = test_inputs[i % len(test_inputs)]
            
            start_time = time.time()
            
            # Create instrumentation for this request
            instrument = bt.create_agent_instrument(
                agent_id=hash(f"benchmark_{model}") % 10000,
                client=self.telemetry_client
            )
            
            session = instrument.start()
            
            try:
                # Simulate AI processing with realistic variations
                processing_time = np.random.exponential(0.5)  # Exponential distribution for realistic latency
                
                if DEMO_MODE:
                    time.sleep(processing_time)
                    
                    # Generate varied outputs for drift analysis
                    base_responses = {
                        "quantum": "Quantum computing uses quantum mechanics principles to process information...",
                        "fibonacci": "def fibonacci(n): return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)",
                        "ai healthcare": "AI is transforming healthcare through diagnostic assistance, drug discovery...",
                        "machine learning": "Machine learning involves training algorithms on data to make predictions...",
                        "api design": "Good API design follows REST principles, uses clear naming conventions..."
                    }
                    
                    # Find matching response
                    output = "AI generated response with detailed analysis and insights."
                    for key, response in base_responses.items():
                        if any(word in input_text.lower() for word in key.split()):
                            # Add variation for drift analysis
                            variations = [
                                response,
                                response.replace(".", "!"),
                                f"{response} Additional context and explanation provided."
                            ]
                            output = np.random.choice(variations)
                            break
                
                latency = time.time() - start_time
                
                # Calculate cost
                cost_estimate = bt.estimate_cost(model, input_text, output)
                cost = cost_estimate.total_cost if cost_estimate else np.random.uniform(0.001, 0.01)
                
                # Simulate accuracy (with some variation)
                base_accuracy = 0.85 if "gpt-4" in model else 0.80
                accuracy = base_accuracy + np.random.uniform(-0.1, 0.15)
                accuracy = max(0.0, min(1.0, accuracy))  # Clamp to [0, 1]
                
                # Update session
                session.set_input_output(input_text, output)
                session.set_model_info(model, temperature=0.1)
                session.set_accuracy(accuracy)
                session.set_cost(cost)
                session.set_metadata("benchmark_run", "true")
                session.set_metadata("request_id", str(i))
                session.add_reasoning_step(f"Benchmark request {i+1}/{num_requests}")
                
                latencies.append(latency)
                costs.append(cost)
                accuracies.append(accuracy)
                outputs.append(output)
                
                # Update Prometheus metrics
                ai_request_duration.labels(
                    model=model,
                    agent_type="benchmark"
                ).observe(latency)
                
                ai_cost_total.labels(
                    model=model,
                    agent_type="benchmark"
                ).inc(cost)
                
            except Exception as e:
                session.set_error(str(e))
                session.set_accuracy(0.0)
                print(f"    Error in request {i}: {e}")
            
            finally:
                session.finish()
        
        # Calculate drift metrics for this model
        drift_metrics = bt.calculate_drift(outputs)
        
        ai_drift_score.labels(
            model=model,
            agent_type="benchmark"
        ).set(drift_metrics.consistency_score)
        
        return {
            "latencies": latencies,
            "costs": costs,
            "accuracies": accuracies,
            "outputs": outputs[:5],  # Store sample outputs
            "drift_metrics": {
                "consistency_score": drift_metrics.consistency_score,
                "agreement_rate": drift_metrics.total_agreement_rate,
                "consensus_confidence": drift_metrics.consensus_confidence
            },
            "stats": {
                "p50_latency": np.percentile(latencies, 50),
                "p95_latency": np.percentile(latencies, 95),
                "p99_latency": np.percentile(latencies, 99),
                "mean_latency": np.mean(latencies),
                "std_latency": np.std(latencies),
                "total_cost": sum(costs),
                "mean_cost": np.mean(costs),
                "mean_accuracy": np.mean(accuracies),
                "min_accuracy": min(accuracies),
                "max_accuracy": max(accuracies)
            }
        }
    
    def _calculate_summary_stats(self, models_data: Dict) -> Dict:
        """Calculate summary statistics across all models."""
        all_latencies = []
        all_costs = []
        all_accuracies = []
        
        model_comparison = {}
        
        for model, data in models_data.items():
            all_latencies.extend(data["latencies"])
            all_costs.extend(data["costs"])
            all_accuracies.extend(data["accuracies"])
            
            model_comparison[model] = {
                "p95_latency": data["stats"]["p95_latency"],
                "mean_cost": data["stats"]["mean_cost"],
                "mean_accuracy": data["stats"]["mean_accuracy"],
                "consistency_score": data["drift_metrics"]["consistency_score"]
            }
        
        # Find best model for each metric
        best_latency = min(model_comparison.keys(), key=lambda m: model_comparison[m]["p95_latency"])
        best_cost = min(model_comparison.keys(), key=lambda m: model_comparison[m]["mean_cost"])
        best_accuracy = max(model_comparison.keys(), key=lambda m: model_comparison[m]["mean_accuracy"])
        best_consistency = max(model_comparison.keys(), key=lambda m: model_comparison[m]["consistency_score"])
        
        return {
            "overall_p95_latency": np.percentile(all_latencies, 95),
            "overall_mean_cost": np.mean(all_costs),
            "overall_mean_accuracy": np.mean(all_accuracies),
            "model_comparison": model_comparison,
            "best_models": {
                "latency": best_latency,
                "cost": best_cost,
                "accuracy": best_accuracy,
                "consistency": best_consistency
            }
        }
    
    def generate_performance_report(self) -> str:
        """Generate a detailed performance report."""
        if not self.benchmark_results:
            return "No benchmark results available."
        
        latest_result = self.benchmark_results[-1]
        summary = latest_result["summary"]
        
        report = f"""
📊 AI Model Performance Benchmark Report
========================================

Test Configuration:
- Timestamp: {latest_result['timestamp'].strftime('%Y-%m-%d %H:%M:%S')}
- Requests per model: {latest_result['num_requests']}
- Models tested: {', '.join(latest_result['models'].keys())}

🚀 Performance Summary:
- Overall P95 Latency: {summary['overall_p95_latency']:.3f}s
- Overall Mean Cost: ${summary['overall_mean_cost']:.6f}
- Overall Mean Accuracy: {summary['overall_mean_accuracy']:.1%}

🏆 Best Performing Models:
- Fastest (P95 Latency): {summary['best_models']['latency']}
- Most Cost-Effective: {summary['best_models']['cost']}
- Highest Accuracy: {summary['best_models']['accuracy']}
- Most Consistent: {summary['best_models']['consistency']}

📈 Model Comparison:
"""
        
        for model, metrics in summary["model_comparison"].items():
            report += f"""
{model}:
  - P95 Latency: {metrics['p95_latency']:.3f}s
  - Mean Cost: ${metrics['mean_cost']:.6f}
  - Mean Accuracy: {metrics['mean_accuracy']:.1%}
  - Consistency Score: {metrics['consistency_score']:.1f}%
"""
        
        return report

# Initialize benchmark system
benchmark = P95Benchmark(client)
print("🎯 P95 Benchmark system initialized")

## 📊 Grafana JSON Export

Generate JSON data compatible with Grafana dashboards:

In [ ]:
class GrafanaJSONExporter:
    """Export telemetry data in Grafana-compatible JSON format."""
    
    def __init__(self):
        self.metrics_data = []
    
    def collect_metrics_snapshot(self) -> Dict:
        """Collect current metrics snapshot."""
        timestamp = int(time.time() * 1000)  # Grafana expects milliseconds
        
        # Collect Prometheus metrics
        prometheus_metrics = generate_latest().decode('utf-8')
        
        # Parse metrics (simplified parsing for demo)
        metrics = {}
        for line in prometheus_metrics.split('\n'):
            if line.startswith('ai_') and not line.startswith('#'):
                parts = line.split(' ')
                if len(parts) >= 2:
                    metric_name = parts[0].split('{')[0]
                    value = float(parts[1])
                    
                    if metric_name not in metrics:
                        metrics[metric_name] = []
                    
                    metrics[metric_name].append({
                        "timestamp": timestamp,
                        "value": value
                    })
        
        snapshot = {
            "timestamp": timestamp,
            "metrics": metrics
        }
        
        self.metrics_data.append(snapshot)
        
        return snapshot
    
    def export_grafana_dashboard(self) -> Dict:
        """Export complete Grafana dashboard configuration."""
        
        dashboard = {
            "dashboard": {
                "id": None,
                "title": "Briefcase AI Telemetry Dashboard",
                "tags": ["ai", "telemetry", "monitoring"],
                "timezone": "browser",
                "refresh": "30s",
                "time": {
                    "from": "now-1h",
                    "to": "now"
                },
                "panels": [
                    {
                        "id": 1,
                        "title": "AI Request Rate",
                        "type": "graph",
                        "gridPos": {"h": 8, "w": 12, "x": 0, "y": 0},
                        "targets": [
                            {
                                "expr": "rate(ai_requests_total[5m])",
                                "legendFormat": "{{model}} - {{status}}",
                                "refId": "A"
                            }
                        ],
                        "yAxes": [
                            {
                                "label": "Requests/second",
                                "min": 0
                            }
                        ]
                    },
                    {
                        "id": 2,
                        "title": "P95 Response Time",
                        "type": "graph",
                        "gridPos": {"h": 8, "w": 12, "x": 12, "y": 0},
                        "targets": [
                            {
                                "expr": "histogram_quantile(0.95, ai_request_duration_seconds_bucket)",
                                "legendFormat": "{{model}} P95",
                                "refId": "A"
                            },
                            {
                                "expr": "histogram_quantile(0.50, ai_request_duration_seconds_bucket)",
                                "legendFormat": "{{model}} P50",
                                "refId": "B"
                            }
                        ],
                        "yAxes": [
                            {
                                "label": "Seconds",
                                "min": 0
                            }
                        ]
                    },
                    {
                        "id": 3,
                        "title": "AI Cost per Hour",
                        "type": "singlestat",
                        "gridPos": {"h": 4, "w": 6, "x": 0, "y": 8},
                        "targets": [
                            {
                                "expr": "rate(ai_cost_usd_total[1h]) * 3600",
                                "refId": "A"
                            }
                        ],
                        "format": "currency",
                        "prefix": "$",
                        "postfix": "/hr"
                    },
                    {
                        "id": 4,
                        "title": "Model Accuracy",
                        "type": "singlestat",
                        "gridPos": {"h": 4, "w": 6, "x": 6, "y": 8},
                        "targets": [
                            {
                                "expr": "ai_accuracy_score",
                                "refId": "A"
                            }
                        ],
                        "format": "percent",
                        "thresholds": "0.7,0.9",
                        "colorBackground": True
                    },
                    {
                        "id": 5,
                        "title": "Drift Consistency Score",
                        "type": "gauge",
                        "gridPos": {"h": 4, "w": 6, "x": 12, "y": 8},
                        "targets": [
                            {
                                "expr": "ai_drift_consistency_score",
                                "refId": "A"
                            }
                        ],
                        "fieldConfig": {
                            "defaults": {
                                "min": 0,
                                "max": 100,
                                "thresholds": {
                                    "steps": [
                                        {"color": "red", "value": 0},
                                        {"color": "yellow", "value": 70},
                                        {"color": "green", "value": 90}
                                    ]
                                }
                            }
                        }
                    },
                    {
                        "id": 6,
                        "title": "Token Usage",
                        "type": "graph",
                        "gridPos": {"h": 8, "w": 12, "x": 0, "y": 12},
                        "targets": [
                            {
                                "expr": "rate(ai_tokens_total[5m])",
                                "legendFormat": "{{model}} - {{token_type}}",
                                "refId": "A"
                            }
                        ],
                        "yAxes": [
                            {
                                "label": "Tokens/second",
                                "min": 0
                            }
                        ]
                    },
                    {
                        "id": 7,
                        "title": "Error Rate",
                        "type": "graph",
                        "gridPos": {"h": 8, "w": 12, "x": 12, "y": 12},
                        "targets": [
                            {
                                "expr": "rate(ai_requests_total{status=\"error\"}[5m]) / rate(ai_requests_total[5m])",
                                "legendFormat": "{{model}} Error Rate",
                                "refId": "A"
                            }
                        ],
                        "yAxes": [
                            {
                                "label": "Error Rate",
                                "min": 0,
                                "max": 1
                            }
                        ]
                    }
                ]
            },
            "folderId": 0,
            "overwrite": True
        }
        
        return dashboard
    
    def export_time_series_data(self, hours_back: int = 1) -> Dict:
        """Export time series data for Grafana JSON data source."""
        
        # Generate sample time series data
        end_time = datetime.now()
        start_time = end_time - timedelta(hours=hours_back)
        
        # Generate data points every minute
        time_points = []
        current_time = start_time
        while current_time <= end_time:
            time_points.append(current_time)
            current_time += timedelta(minutes=1)
        
        # Create sample data for each metric
        time_series = {
            "ai_requests_per_minute": [],
            "ai_p95_latency": [],
            "ai_cost_per_hour": [],
            "ai_accuracy": [],
            "ai_consistency_score": []
        }
        
        for i, timestamp in enumerate(time_points):
            ts = int(timestamp.timestamp() * 1000)  # Grafana expects milliseconds
            
            # Simulate realistic data with trends
            base_requests = 10 + 5 * np.sin(i * 0.1) + np.random.normal(0, 2)
            base_latency = 0.5 + 0.2 * np.sin(i * 0.05) + np.random.normal(0, 0.1)
            base_cost = 0.01 + 0.005 * np.sin(i * 0.08) + np.random.normal(0, 0.002)
            base_accuracy = 0.9 + 0.05 * np.sin(i * 0.03) + np.random.normal(0, 0.02)
            base_consistency = 85 + 10 * np.sin(i * 0.02) + np.random.normal(0, 3)
            
            time_series["ai_requests_per_minute"].append([base_requests, ts])
            time_series["ai_p95_latency"].append([max(0, base_latency), ts])
            time_series["ai_cost_per_hour"].append([max(0, base_cost), ts])
            time_series["ai_accuracy"].append([max(0, min(1, base_accuracy)), ts])
            time_series["ai_consistency_score"].append([max(0, min(100, base_consistency)), ts])
        
        return time_series
    
    def save_dashboard_json(self, filename: str = "grafana_dashboard.json"):
        """Save Grafana dashboard configuration to file."""
        dashboard = self.export_grafana_dashboard()
        
        with open(filename, 'w') as f:
            json.dump(dashboard, f, indent=2)
        
        print(f"📊 Grafana dashboard exported to {filename}")
        return filename
    
    def save_time_series_json(self, filename: str = "time_series_data.json", hours_back: int = 1):
        """Save time series data to file."""
        time_series = self.export_time_series_data(hours_back)
        
        with open(filename, 'w') as f:
            json.dump(time_series, f, indent=2)
        
        print(f"📈 Time series data exported to {filename}")
        return filename

# Initialize Grafana exporter
grafana_exporter = GrafanaJSONExporter()
print("📊 Grafana JSON exporter initialized")

## 🚀 Running the End-to-End Demo

Now let's run the complete workflow:

### Step 1: Run Replay System

In [ ]:
# Run replay system with sample interactions
print("🎬 Running Replay System Demo...")
replay_results = replay_loader.run_replay_batch(max_interactions=5)

# Analyze replay results
successful_replays = [r for r in replay_results if r["status"] == "success"]
avg_accuracy = np.mean([r["accuracy"] for r in successful_replays]) if successful_replays else 0
total_cost = sum([r["cost"] for r in successful_replays]) if successful_replays else 0

print(f"\n📊 Replay Results:")
print(f"   Successful replays: {len(successful_replays)}/{len(replay_results)}")
print(f"   Average accuracy: {avg_accuracy:.1%}")
print(f"   Total cost: ${total_cost:.6f}")

# Show sample outputs
print(f"\n🔍 Sample Replay Output:")
if successful_replays:
    sample = successful_replays[0]
    print(f"   Input: {sample['interaction'].input_text[:60]}...")
    print(f"   Output: {sample['actual_output'][:80]}...")
    print(f"   Accuracy: {sample['accuracy']:.1%}")

### Step 2: LangChain Agent Demo

In [ ]:
# Create and test LangChain agent with telemetry
print("🦜 Running LangChain Integration Demo...")

agent, callback_handler = create_monitored_langchain_agent(agent_id=1001)

test_queries = [
    "Analyze the sales data for Q3 and identify key trends",
    "Query the database for customer information and generate insights",
    "Help me understand the performance metrics for our AI models"
]

langchain_results = []
for i, query in enumerate(test_queries):
    print(f"\n🔍 Query {i+1}: {query[:50]}...")
    try:
        response = agent.run(query)
        langchain_results.append({
            "query": query,
            "response": response,
            "status": "success"
        })
        print(f"   ✅ Response: {response[:60]}...")
    except Exception as e:
        langchain_results.append({
            "query": query,
            "error": str(e),
            "status": "error"
        })
        print(f"   ❌ Error: {str(e)[:50]}...")

successful_queries = [r for r in langchain_results if r["status"] == "success"]
print(f"\n📊 LangChain Results: {len(successful_queries)}/{len(test_queries)} successful")

### Step 3: P95 Performance Benchmark

In [ ]:
# Run P95 performance benchmark
print("🎯 Running P95 Performance Benchmark...")

models_to_test = ["gpt-3.5-turbo", "gpt-4"]
if DEMO_MODE:
    models_to_test.append("claude-3-sonnet")

benchmark_results = benchmark.run_performance_benchmark(
    num_requests=20,  # Reduced for demo
    models=models_to_test
)

# Display benchmark report
print(benchmark.generate_performance_report())

### Step 4: Drift Analysis Across All Runs

In [ ]:
# Analyze drift across all collected outputs
print("📈 Running Comprehensive Drift Analysis...")

all_outputs = []

# Collect outputs from replay
replay_outputs = [r["actual_output"] for r in successful_replays]
all_outputs.extend(replay_outputs)

# Collect outputs from LangChain
langchain_outputs = [r["response"] for r in successful_queries]
all_outputs.extend(langchain_outputs)

# Collect outputs from benchmark (sample)
for model_data in benchmark_results["models"].values():
    all_outputs.extend(model_data["outputs"][:3])  # Sample outputs

if all_outputs:
    # Calculate comprehensive drift metrics
    overall_drift = bt.calculate_drift(all_outputs)
    
    print(f"\n🔍 Overall Drift Analysis:")
    print(f"   Total outputs analyzed: {len(all_outputs)}")
    print(f"   Agreement rate: {overall_drift.total_agreement_rate:.1f}%")
    print(f"   Consistency score: {overall_drift.consistency_score:.1f}")
    print(f"   Consensus confidence: {overall_drift.consensus_confidence}")
    print(f"   Factual drift count: {overall_drift.factual_drift_count}")
    
    # Check for concerning drift
    if overall_drift.consensus_confidence == "low":
        print("   ⚠️ WARNING: Low consensus detected across AI outputs!")
    elif overall_drift.consensus_confidence == "medium":
        print("   ⚡ NOTICE: Medium consensus - monitor for trends")
    else:
        print("   ✅ GOOD: High consensus across AI outputs")
else:
    print("No outputs available for drift analysis")

### Step 5: Export Prometheus Metrics

In [ ]:
# Export current Prometheus metrics
print("📊 Exporting Prometheus Metrics...")

prometheus_output = generate_latest().decode('utf-8')

# Save to file
with open('prometheus_metrics.txt', 'w') as f:
    f.write(prometheus_output)

print("✅ Prometheus metrics exported to prometheus_metrics.txt")

# Show key metrics
print("\n🔍 Key Prometheus Metrics:")
key_metrics = ['ai_requests_total', 'ai_request_duration_seconds', 'ai_cost_usd_total']

for metric in key_metrics:
    metric_lines = [line for line in prometheus_output.split('\n') if line.startswith(metric) and not line.startswith('#')]
    if metric_lines:
        print(f"   {metric}: {len(metric_lines)} series")
        # Show first metric value as example
        if metric_lines:
            sample = metric_lines[0]
            print(f"     Example: {sample[:80]}...")

### Step 6: Generate Grafana Dashboard

In [ ]:
# Collect current metrics and export Grafana JSON
print("📊 Generating Grafana Dashboard...")

# Collect metrics snapshot
metrics_snapshot = grafana_exporter.collect_metrics_snapshot()
print(f"   Collected metrics at {datetime.fromtimestamp(metrics_snapshot['timestamp']/1000)}")

# Export dashboard configuration
dashboard_file = grafana_exporter.save_dashboard_json("ai_telemetry_dashboard.json")

# Export time series data
time_series_file = grafana_exporter.save_time_series_json("ai_metrics_data.json", hours_back=2)

print(f"\n📈 Grafana Files Created:")
print(f"   Dashboard config: {dashboard_file}")
print(f"   Time series data: {time_series_file}")
print(f"\n💡 To use with Grafana:")
print(f"   1. Import {dashboard_file} as a new dashboard")
print(f"   2. Configure Prometheus data source pointing to your metrics endpoint")
print(f"   3. Use {time_series_file} with JSON data source for historical data")

## 📊 Data Analysis and Visualization

Let's analyze the collected data with pandas and matplotlib:

In [ ]:
# Create comprehensive analysis of all collected data
print("📈 Creating Data Analysis Dashboard...")

# Prepare data for analysis
analysis_data = []

# Add replay data
for result in successful_replays:
    analysis_data.append({
        "source": "replay",
        "agent_type": result["interaction"].agent_id,
        "model": result["interaction"].model,
        "accuracy": result["accuracy"],
        "cost": result["cost"],
        "duration": result["duration"],
        "timestamp": result["interaction"].timestamp
    })

# Add benchmark data
for model, model_data in benchmark_results["models"].items():
    for i, (latency, cost, accuracy) in enumerate(zip(
        model_data["latencies"], 
        model_data["costs"], 
        model_data["accuracies"]
    )):
        analysis_data.append({
            "source": "benchmark",
            "agent_type": "benchmark",
            "model": model,
            "accuracy": accuracy,
            "cost": cost,
            "duration": latency,
            "timestamp": datetime.now() - timedelta(minutes=i)
        })

# Create DataFrame
df = pd.DataFrame(analysis_data)

if not df.empty:
    print(f"📊 Analysis dataset: {len(df)} records")
    print(f"   Sources: {df['source'].unique()}")
    print(f"   Models: {df['model'].unique()}")
    print(f"   Agent types: {df['agent_type'].unique()}")
    
    # Create visualizations
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('AI Telemetry Analysis Dashboard', fontsize=16, fontweight='bold')
    
    # 1. Accuracy by Model
    df.boxplot(column='accuracy', by='model', ax=axes[0, 0])
    axes[0, 0].set_title('Accuracy Distribution by Model')
    axes[0, 0].set_xlabel('Model')
    axes[0, 0].set_ylabel('Accuracy')
    
    # 2. Cost vs Duration
    for model in df['model'].unique():
        model_data = df[df['model'] == model]
        axes[0, 1].scatter(model_data['duration'], model_data['cost'], 
                          label=model, alpha=0.7, s=50)
    axes[0, 1].set_xlabel('Duration (seconds)')
    axes[0, 1].set_ylabel('Cost ($)')
    axes[0, 1].set_title('Cost vs Duration by Model')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Performance by Source
    source_stats = df.groupby(['source', 'model']).agg({
        'accuracy': 'mean',
        'cost': 'mean',
        'duration': 'mean'
    }).reset_index()
    
    sns.barplot(data=source_stats, x='source', y='accuracy', hue='model', ax=axes[0, 2])
    axes[0, 2].set_title('Average Accuracy by Source and Model')
    axes[0, 2].set_ylabel('Average Accuracy')
    
    # 4. Cost Distribution
    df['cost'].hist(bins=20, alpha=0.7, ax=axes[1, 0])
    axes[1, 0].axvline(df['cost'].mean(), color='red', linestyle='--', 
                       label=f'Mean: ${df["cost"].mean():.6f}')
    axes[1, 0].axvline(df['cost'].median(), color='green', linestyle='--',
                       label=f'Median: ${df["cost"].median():.6f}')
    axes[1, 0].set_title('Cost Distribution')
    axes[1, 0].set_xlabel('Cost ($)')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 5. Duration vs Accuracy
    axes[1, 1].scatter(df['duration'], df['accuracy'], alpha=0.7, c=df['cost'], 
                       cmap='viridis', s=50)
    axes[1, 1].set_xlabel('Duration (seconds)')
    axes[1, 1].set_ylabel('Accuracy')
    axes[1, 1].set_title('Duration vs Accuracy (colored by cost)')
    axes[1, 1].grid(True, alpha=0.3)
    
    # Add colorbar for cost
    cbar = plt.colorbar(axes[1, 1].collections[0], ax=axes[1, 1])
    cbar.set_label('Cost ($)')
    
    # 6. Summary Statistics Table
    axes[1, 2].axis('off')
    summary_stats = df.groupby('model').agg({
        'accuracy': ['mean', 'std'],
        'cost': ['mean', 'sum'],
        'duration': ['mean', lambda x: np.percentile(x, 95)]
    }).round(6)
    
    # Flatten column names
    summary_stats.columns = ['Acc_Mean', 'Acc_Std', 'Cost_Mean', 'Cost_Total', 'Dur_Mean', 'Dur_P95']
    
    # Create table
    table_data = summary_stats.reset_index().values
    table = axes[1, 2].table(cellText=table_data, 
                            colLabels=['Model', 'Acc Mean', 'Acc Std', 'Cost Mean', 'Cost Total', 'Dur Mean', 'Dur P95'],
                            cellLoc='center', loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1.2, 1.5)
    axes[1, 2].set_title('Model Performance Summary')
    
    plt.tight_layout()
    plt.savefig('ai_telemetry_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print summary statistics
    print("\n📊 Summary Statistics:")
    print(f"   Total requests analyzed: {len(df)}")
    print(f"   Average accuracy: {df['accuracy'].mean():.1%}")
    print(f"   Total cost: ${df['cost'].sum():.6f}")
    print(f"   Average duration: {df['duration'].mean():.3f}s")
    print(f"   P95 duration: {np.percentile(df['duration'], 95):.3f}s")
    
    # Model comparison
    print("\n🏆 Model Performance Ranking:")
    model_ranking = df.groupby('model').agg({
        'accuracy': 'mean',
        'cost': 'mean',
        'duration': lambda x: np.percentile(x, 95)
    }).round(6)
    
    print("   By Accuracy (highest first):")
    for model, acc in model_ranking.sort_values('accuracy', ascending=False)['accuracy'].items():
        print(f"     {model}: {acc:.1%}")
    
    print("   By Cost (lowest first):")
    for model, cost in model_ranking.sort_values('cost')['cost'].items():
        print(f"     {model}: ${cost:.6f}")
    
    print("   By P95 Latency (lowest first):")
    for model, dur in model_ranking.sort_values('duration')['duration'].items():
        print(f"     {model}: {dur:.3f}s")

else:
    print("❌ No data available for analysis")

## 🔧 Complete Integration Example

Here's how you would integrate everything in a production environment:

In [ ]:
# Production integration example
class ProductionAITelemetryHarness:
    """Complete AI telemetry harness for production use."""
    
    def __init__(self, briefcase_api_key: str, openai_api_key: str = None):
        # Core telemetry
        self.telemetry_client = bt.create_client(briefcase_api_key)
        self.telemetry_client.start_background_flush()
        
        # Monitoring systems
        self.replay_loader = ReplayLoader(self.telemetry_client)
        self.benchmark = P95Benchmark(self.telemetry_client)
        self.grafana_exporter = GrafanaJSONExporter()
        
        # LangChain setup
        if openai_api_key:
            self.llm = ChatOpenAI(openai_api_key=openai_api_key)
        
        # Performance tracking
        self.performance_history = []
        self.drift_alerts = []
        
    def create_monitored_agent(self, agent_id: int, tools: List = None):
        """Create a LangChain agent with full telemetry monitoring."""
        callback_handler = TelemetryCallbackHandler(self.telemetry_client, agent_id)
        
        if hasattr(self, 'llm'):
            agent = initialize_agent(
                tools or [],
                self.llm,
                agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
                callbacks=[callback_handler]
            )
        else:
            # Use mock agent for demo
            agent, _ = create_monitored_langchain_agent(agent_id)
        
        return agent
    
    def run_comprehensive_monitoring(self):
        """Run comprehensive monitoring workflow."""
        print("🔄 Starting comprehensive monitoring workflow...")
        
        # 1. Load and replay historical interactions
        print("\n1️⃣ Running replay validation...")
        replay_results = self.replay_loader.run_replay_batch(max_interactions=3)
        
        # 2. Run performance benchmarks
        print("\n2️⃣ Running performance benchmarks...")
        benchmark_results = self.benchmark.run_performance_benchmark(num_requests=10)
        
        # 3. Analyze drift across all outputs
        print("\n3️⃣ Analyzing model drift...")
        all_outputs = []
        
        # Collect outputs from various sources
        successful_replays = [r for r in replay_results if r["status"] == "success"]
        all_outputs.extend([r["actual_output"] for r in successful_replays])
        
        for model_data in benchmark_results["models"].values():
            all_outputs.extend(model_data["outputs"][:2])
        
        if all_outputs:
            drift_metrics = bt.calculate_drift(all_outputs)
            
            # Check for drift alerts
            if drift_metrics.consensus_confidence == "low":
                self.drift_alerts.append({
                    "timestamp": datetime.now(),
                    "severity": "high",
                    "message": f"Low consensus detected: {drift_metrics.total_agreement_rate:.1f}% agreement",
                    "metrics": drift_metrics
                })
            
            print(f"   Drift analysis complete: {drift_metrics.consensus_confidence} confidence")
        
        # 4. Export monitoring data
        print("\n4️⃣ Exporting monitoring data...")
        self.grafana_exporter.collect_metrics_snapshot()
        dashboard_file = self.grafana_exporter.save_dashboard_json()
        metrics_file = self.grafana_exporter.save_time_series_json()
        
        # 5. Generate summary report
        print("\n5️⃣ Generating summary report...")
        return self.generate_monitoring_report({
            "replay_results": replay_results,
            "benchmark_results": benchmark_results,
            "drift_metrics": drift_metrics if all_outputs else None,
            "dashboard_file": dashboard_file,
            "metrics_file": metrics_file
        })
    
    def generate_monitoring_report(self, monitoring_data: Dict) -> str:
        """Generate comprehensive monitoring report."""
        
        report = f"""
🚀 AI TELEMETRY COMPREHENSIVE MONITORING REPORT
==============================================
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

📊 EXECUTIVE SUMMARY
-------------------
"""
        
        # Replay summary
        replay_results = monitoring_data["replay_results"]
        successful_replays = [r for r in replay_results if r["status"] == "success"]
        if successful_replays:
            avg_accuracy = np.mean([r["accuracy"] for r in successful_replays])
            total_cost = sum([r["cost"] for r in successful_replays])
            report += f"""
Replay Validation:
✓ {len(successful_replays)}/{len(replay_results)} successful replays
✓ Average accuracy: {avg_accuracy:.1%}
✓ Total cost: ${total_cost:.6f}
"""
        
        # Benchmark summary
        benchmark_results = monitoring_data["benchmark_results"]
        summary = benchmark_results["summary"]
        report += f"""
Performance Benchmarks:
✓ Overall P95 latency: {summary['overall_p95_latency']:.3f}s
✓ Overall mean cost: ${summary['overall_mean_cost']:.6f}
✓ Best latency model: {summary['best_models']['latency']}
✓ Most cost-effective: {summary['best_models']['cost']}
"""
        
        # Drift analysis
        drift_metrics = monitoring_data["drift_metrics"]
        if drift_metrics:
            status_emoji = "✅" if drift_metrics.consensus_confidence == "high" else "⚠️" if drift_metrics.consensus_confidence == "medium" else "🚨"
            report += f"""
Drift Analysis:
{status_emoji} Consensus confidence: {drift_metrics.consensus_confidence}
{status_emoji} Agreement rate: {drift_metrics.total_agreement_rate:.1f}%
{status_emoji} Consistency score: {drift_metrics.consistency_score:.1f}
"""
        
        # Alerts
        if self.drift_alerts:
            report += f"""
🚨 ACTIVE ALERTS ({len(self.drift_alerts)}):
"""
            for alert in self.drift_alerts[-3:]:  # Show last 3 alerts
                report += f"⚠️ {alert['timestamp'].strftime('%H:%M:%S')}: {alert['message']}\n"
        
        # Monitoring files
        report += f"""
📁 GENERATED FILES:
------------------
📊 Grafana Dashboard: {monitoring_data['dashboard_file']}
📈 Time Series Data: {monitoring_data['metrics_file']}
📊 Analysis Chart: ai_telemetry_analysis.png
🔧 Prometheus Metrics: prometheus_metrics.txt

🔗 NEXT STEPS:
-------------
1. Import Grafana dashboard for real-time monitoring
2. Set up Prometheus scraping for continuous metrics collection
3. Configure alerting rules for drift detection
4. Schedule regular benchmark runs for performance tracking
5. Review and validate any drift alerts

For detailed analysis, review the generated charts and data files.
"""
        
        return report

# Demo the complete production harness
print("🏭 Initializing Production AI Telemetry Harness...")
harness = ProductionAITelemetryHarness(
    briefcase_api_key="demo-key",
    openai_api_key=OPENAI_API_KEY if not DEMO_MODE else None
)

# Run comprehensive monitoring
monitoring_report = harness.run_comprehensive_monitoring()

print(monitoring_report)

## 📋 Final Summary and Next Steps

This notebook demonstrated a complete end-to-end AI telemetry workflow:

In [ ]:
# Final summary
print("""
🎉 END-TO-END DEMO COMPLETED SUCCESSFULLY!
==========================================

✅ FEATURES DEMONSTRATED:
------------------------
🎬 Replay System: Captured and replayed AI interactions for validation
🦜 LangChain Integration: Monitored LangChain agents with full telemetry
📊 Prometheus Metrics: Exported comprehensive performance metrics
📈 Grafana Dashboard: Generated ready-to-use dashboard configuration
🎯 P95 Benchmarking: Measured performance across multiple models
📊 Drift Analysis: Detected consistency patterns across AI outputs
💰 Cost Tracking: Monitored and optimized AI usage costs
⚖️ Compliance Ready: Framework for regulatory compliance monitoring

📁 FILES GENERATED:
------------------
• ai_telemetry_dashboard.json - Grafana dashboard configuration
• ai_metrics_data.json - Time series data for visualization
• prometheus_metrics.txt - Current Prometheus metrics snapshot
• ai_telemetry_analysis.png - Comprehensive analysis charts

🚀 PRODUCTION DEPLOYMENT:
------------------------
1. Set up Prometheus server to scrape metrics endpoint
2. Import Grafana dashboard for real-time monitoring
3. Configure alerting rules for drift and performance thresholds
4. Schedule regular benchmark runs via cron/scheduler
5. Set up replay validation in CI/CD pipeline
6. Integrate LangChain callback handlers in production agents
7. Configure compliance monitoring for your regulatory requirements

💡 KEY INSIGHTS:
---------------
• AI telemetry provides crucial visibility into model behavior
• Drift detection helps maintain consistent AI performance
• Cost tracking enables budget optimization and model selection
• Comprehensive monitoring supports production AI reliability
• Integration with existing tools (LangChain, Grafana, Prometheus) is seamless

📚 NEXT STEPS:
-------------
• Explore the examples/ directory for more specific use cases
• Review the docs/ directory for detailed implementation guides
• Check the API reference for advanced configuration options
• Implement custom compliance frameworks for your industry
• Set up automated drift alerting in your monitoring system

Thank you for exploring the Briefcase AI Telemetry SDK! 🚀
""")

# Final telemetry flush
final_buffer_size = client.buffer_size()
if final_buffer_size > 0:
    flushed = client.flush()
    print(f"\n📤 Final flush: {flushed} events sent to telemetry service")

print(f"\n📊 Session Complete - Check your generated files for detailed analysis!")